![Foundations of NLP] Embeddings()

---



### About:  
Computers are only able to work with numerical data. One of the most important tasks for working with language is converting the text data into a numerical form. Text embeddings aim to preserve the meaning and context of words by assigning a vector to words where similar words with similar meanings or contexts would have similar vectors. In this notebook we will explore progressively more complex text embeddings and talk about how they are used. 


### Learning Objective:
1. Compare two embedding methods, Word2Vec and BERT, using customer reviews


### Prerequisites:
- Python 
- Core data science libraries Numpy and Pandas 
- Understand, run, and evaluate classification models in Scikit Learn 

### Installs
- [gensim](https://radimrehurek.com/gensim/intro.html#installation)



### Imports

In [1]:
#imports 
import numpy as np
import pandas as pd

## Gensim for word2vec 
# to clean text and implement word2vec imports 
from gensim.parsing.preprocessing import remove_stopwords
from gensim.utils import simple_preprocess
from gensim.models import  Word2Vec

## Pytorch imports for BERT embeddings
from transformers import AutoTokenizer, AutoModel

## Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity



Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


### NLP Scenario 
You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product.

#####  Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."


In [2]:
# data setup 
# create a list of reviews 
reviews = ["I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!",
"I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.",
"The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
"I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
"The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.",
"The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."
]

## Static Embeddings 

We will use the Gensim implementation of Word2Vec on our reviews to explore static embeddings. 

Gensim is a popular open-source library for natural language processing. It excels at topic modeling and similarity methods using models such as Latent Dirichlet Allocation (LDA) and Word2Vec. We will use Word2Vec to find the similarity across our reviews. 

### Tokenization and pre-processing
In order to use the Word2Vec model we first need to convert our raw text to tokens. 
We will do this with a function called "preprocess_text" that using built-in preprocessing tools from Gensim to:
1. make all the words lowercase 
2. remove special characters 
3. remove very common words, known as stop words 

After running this function we will create a list that contains cleaned tokens for each review. These tokens will be used in our Word2Vec model. 

In [28]:
def preprocess_text(text):
    """
    Clean and tokenize text using gensim's preprocessing utilities
    """
    # Convert to lowercase and complete other basic cleaning 
    tokens = simple_preprocess(text)  

    # Remove stopwords
    clean_text = remove_stopwords(' '.join(tokens))
    tokens = clean_text.split()
    
    return tokens

# Preprocess the reviews by calling our preprocess_text function on each review and saving to a new list
processed_reviews = [preprocess_text(review) for review in reviews]

In [4]:
#lets review our tokenized reviews
for review in processed_reviews:
    print(review)


['absolutely', 'love', 'techwave', 'daily', 'tasks', 'easier', 'efficient', 'highly', 'recommend']
['impressed', 'techwave', 'lacks', 'essential', 'features', 'slow']
['techwave', 'fantastic', 'exceeded', 'expectations', 'essential', 'daily', 'routine']
['techwave', 'average', 'job', 'particularly', 'special']
['techwave', 'terrible', 'glitches', 'crashes', 'frequently', 'regret', 'purchasing']
['techwave', 'disappointing', 'live', 'hype', 'missing', 'key', 'functionalities']


Compare the tokenized reviews with the original reviews below. Note that the text has been converted to lowercase, special characters and numbers have been removed, and stopwords have been removed.

#### Implementing Word2Vec

Text embeddings aim to preserve the meaning and context of words by assigning a vector to words where similar words with similar meanings or contexts would have similar vectors. 

We will call Gensims's [Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html) to demonstrate one example of a static embedding. 
- Input: 
  - Tokens for each review that have been pre-processed (cleaned)
- Output:
  - A vector (size= 100 here) 


In [33]:
#initialize the word2vec model on our reviews
w2v_model = Word2Vec(sentences=processed_reviews, vector_size=100, min_count=1)

In [34]:
# Get the word vector for a word (impressed)
w2v_model.wv['impressed']

array([-0.00713643,  0.00124286, -0.00717734, -0.00224199,  0.00371802,
        0.00583452,  0.00120224,  0.00210208, -0.00410936,  0.00722499,
       -0.00630799,  0.00464695, -0.00821638,  0.00203776, -0.00497706,
       -0.00424803, -0.0031055 ,  0.00565393,  0.00579901, -0.0049749 ,
        0.00077551, -0.00849458,  0.00780813,  0.00925633, -0.00274683,
        0.00079877,  0.00074392,  0.00547549, -0.00860525,  0.00058607,
        0.00687236,  0.00222927,  0.00112431, -0.00932279,  0.00847827,
       -0.00626235, -0.00299408,  0.00349698, -0.00077333,  0.00140856,
        0.00178389, -0.00682707, -0.00972425,  0.0090419 ,  0.0061968 ,
       -0.00691464,  0.00340141,  0.00020196,  0.0047568 , -0.00711792,
        0.00402528,  0.0043457 ,  0.00995693, -0.00447612, -0.00138816,
       -0.00731262, -0.00969637, -0.00907991, -0.00102291, -0.00650428,
        0.00484872, -0.00616584,  0.00251948,  0.00073846, -0.0033943 ,
       -0.0009763 ,  0.00997766,  0.00914412, -0.00446103,  0.00

In [35]:
w2v_model.wv['terrible']

array([-0.00250953, -0.00590217,  0.0074822 , -0.00725842, -0.00896395,
       -0.00178884, -0.00836537,  0.0003886 ,  0.00194596, -0.00246722,
       -0.00644809, -0.00032226, -0.00109785,  0.00349419,  0.00812514,
        0.00585427,  0.00844402, -0.00897145,  0.00944224, -0.0023653 ,
        0.00869744,  0.0023863 ,  0.00358709, -0.00958228, -0.00948932,
        0.00898206, -0.00289963,  0.00281721,  0.00641873, -0.00029925,
        0.00972494, -0.00103496, -0.00967458, -0.00705534, -0.0010451 ,
       -0.00867644,  0.00742031,  0.00362219, -0.00874588,  0.00847923,
        0.0089295 ,  0.00584687,  0.00690791, -0.0095728 ,  0.00049108,
       -0.00922118, -0.00366706,  0.00025248, -0.00027871,  0.00146658,
        0.00327773,  0.00212536,  0.00532139,  0.00751998, -0.0058855 ,
        0.00795908,  0.005994  ,  0.00978648,  0.00462247, -0.00332757,
       -0.00374825, -0.00063195, -0.00165463,  0.00987211,  0.00112103,
        0.00400728,  0.00341747, -0.00885225,  0.0067198 ,  0.00

In [36]:
# find the most similar words to a word
w2v_model.wv.most_similar('impressed')

[('easier', 0.2528840899467468),
 ('frequently', 0.20094607770442963),
 ('live', 0.17532140016555786),
 ('essential', 0.17016783356666565),
 ('efficient', 0.15017001330852509),
 ('fantastic', 0.1388774812221527),
 ('exceeded', 0.10851150751113892),
 ('particularly', 0.09937429428100586),
 ('absolutely', 0.05292689427733421),
 ('slow', 0.0347055047750473)]

In [37]:
w2v_model.wv.most_similar('terrible')

[('tasks', 0.17134226858615875),
 ('particularly', 0.1566605269908905),
 ('easier', 0.14718976616859436),
 ('disappointing', 0.1326819658279419),
 ('daily', 0.12809142470359802),
 ('efficient', 0.11393295973539352),
 ('recommend', 0.10803277045488358),
 ('special', 0.1075734943151474),
 ('frequently', 0.10151747614145279),
 ('hype', 0.0956375002861023)]

Notice that the word 'impressed' is similar to 'essential' and 'fantastic', while the word 'terrible' is similar to 'hype' and 'disappointing'. Note some words are similar to both (such as 'easier')! 

This shows that the Word2Vec model has learned the some of the semantic relationships between words based on the context in which they appear in the reviews. Given our very small training set, these results are impressive. With more data and more examples of usage for the model to train on, these results would improve. 

## Try it! 
### Evaluate the impact of modifying your vector size 

#### Smaller vector
- Modify the  word2vec model to use a smaller vector size (e.g. 10)
- Then, retrain the model on the processed reviews and find the most similar words to the word "impressed".
- What differences do you notice in the results compared to the previous model?


In [38]:
#initialize the word2vec model on our reviews with a smaller vector size
w2v_model = Word2Vec(sentences=processed_reviews, vector_size=10, window=5, min_count=1)
# find the most similar words to impressed 
w2v_model.wv.most_similar('impressed')

[('slow', 0.6144205331802368),
 ('job', 0.39106518030166626),
 ('live', 0.3549245297908783),
 ('exceeded', 0.34240788221359253),
 ('fantastic', 0.24943944811820984),
 ('special', 0.24015657603740692),
 ('efficient', 0.14173880219459534),
 ('easier', 0.1116168275475502),
 ('purchasing', 0.07380060851573944),
 ('terrible', 0.06157125532627106)]

#### Larger Vector and Window
- Repeat the process and modify the word2vec model to use a larger vector size (e.g. 300) and a larger window size (e.g. 10). 
- What differences do you notice in the results compared to the previous models?

In [39]:
#initialize the word2vec model on our reviews with a smaller vector size
w2v_model = Word2Vec(sentences=processed_reviews, vector_size=300, window=10, min_count=1)
# find the most similar words to impressed 
w2v_model.wv.most_similar('impressed')

[('features', 0.1225011944770813),
 ('routine', 0.12167208641767502),
 ('job', 0.07225648313760757),
 ('fantastic', 0.06857949495315552),
 ('purchasing', 0.06097680330276489),
 ('glitches', 0.054830193519592285),
 ('terrible', 0.0528157539665699),
 ('average', 0.04623639956116676),
 ('crashes', 0.04303780198097229),
 ('daily', 0.04040076211094856)]

Notice that the word2vec model is sensitive to the vector size and other hyperparameters. These choices can affect the quality of the word embeddings and the similarity results. Results are also impacted by your training data. Here we only have a small set of reviews, so the word embeddings may not be as accurate as they would be with a larger dataset.



### Understanding word order 

Let's examine how word order affects meaning with these two sentences:

- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"

The meaning is changed depending on which words come first. Let's first make embeddings using a static embedding (Word2Vec), then compare the difference when we use a contextual embedding. 

## Try it!  
Use the steps above to convert these to vectors using word to vec 

In [40]:
# create a list of sentences
sentences = ["The supplier agreed to pay the manufacturer",
             "The manufacturer agreed to pay the supplier"]


# Preprocess the sentences using the function we created earlier, preprocess_text()
processed_reviews = [preprocess_text(sentence) for sentence in sentences]

# Initialize the Word2Vec model
w2v_model= Word2Vec(sentences=processed_reviews, vector_size=100, window=5, sg=0, min_count=1)


In [41]:
# Get the word vector for a word (supplier)
w2v_model.wv['supplier']

array([-8.2426779e-03,  9.2993546e-03, -1.9766092e-04, -1.9672764e-03,
        4.6036304e-03, -4.0953159e-03,  2.7431143e-03,  6.9399667e-03,
        6.0654259e-03, -7.5107943e-03,  9.3823504e-03,  4.6718083e-03,
        3.9661205e-03, -6.2435055e-03,  8.4599797e-03, -2.1501649e-03,
        8.8251876e-03, -5.3620026e-03, -8.1294188e-03,  6.8245591e-03,
        1.6711927e-03, -2.1985089e-03,  9.5136007e-03,  9.4938548e-03,
       -9.7740470e-03,  2.5052286e-03,  6.1566923e-03,  3.8724565e-03,
        2.0227872e-03,  4.3050171e-04,  6.7363144e-04, -3.8206363e-03,
       -7.1402504e-03, -2.0888723e-03,  3.9238976e-03,  8.8186832e-03,
        9.2591504e-03, -5.9759365e-03, -9.4026709e-03,  9.7643770e-03,
        3.4297847e-03,  5.1661171e-03,  6.2823449e-03, -2.8042626e-03,
        7.3227035e-03,  2.8302716e-03,  2.8710044e-03, -2.3803699e-03,
       -3.1282497e-03, -2.3701417e-03,  4.2764368e-03,  7.6057913e-05,
       -9.5842788e-03, -9.6655441e-03, -6.1481940e-03, -1.2856961e-04,
      

In [42]:
# Get the word vector for a word (manufacturer)
w2v_model.wv['manufacturer']

array([-5.3622725e-04,  2.3643136e-04,  5.1033497e-03,  9.0092728e-03,
       -9.3029495e-03, -7.1168090e-03,  6.4588725e-03,  8.9729885e-03,
       -5.0154282e-03, -3.7633716e-03,  7.3805046e-03, -1.5334714e-03,
       -4.5366134e-03,  6.5540518e-03, -4.8601604e-03, -1.8160177e-03,
        2.8765798e-03,  9.9187379e-04, -8.2852151e-03, -9.4488179e-03,
        7.3117660e-03,  5.0702621e-03,  6.7576934e-03,  7.6286553e-04,
        6.3508903e-03, -3.4053659e-03, -9.4640139e-04,  5.7685734e-03,
       -7.5216377e-03, -3.9361035e-03, -7.5115822e-03, -9.3004224e-04,
        9.5381187e-03, -7.3191668e-03, -2.3337686e-03, -1.9377411e-03,
        8.0774371e-03, -5.9308959e-03,  4.5162440e-05, -4.7537340e-03,
       -9.6035507e-03,  5.0072931e-03, -8.7595852e-03, -4.3918253e-03,
       -3.5099984e-05, -2.9618145e-04, -7.6612402e-03,  9.6147433e-03,
        4.9820580e-03,  9.2331432e-03, -8.1579173e-03,  4.4957981e-03,
       -4.1370760e-03,  8.2453608e-04,  8.4986202e-03, -4.4621765e-03,
      

In [43]:
cos_sim = cosine_similarity([w2v_model.wv['supplier']], [w2v_model.wv['supplier']])
print(cos_sim)

[[0.9999999]]


#### What did you notice? 

These two sentences would have identical representations in word2vec but have very different meanings. In this example, the order of the words is critical to the meaning of the phrase. We aren't able to capture this order with the bag-of-words approach used in word2vec and other similar models. To address this issue, methods were developed to capture the sequential order of words. 


## Contextual Embeddings 

Unlike static embeddings, like word2vec, contextual embeddings create different embeddings for the same word depending on the context in which it was used. To capture the context we use information such as the words around our word of interest, its sentence position, and other factors to map its vector. In other words, contextual embeddings are position aware. 

Let's look at our example from before: 
- "The supplier agreed to pay the manufacturer"
- "The manufacturer agreed to pay the supplier"

We will use BERT, a famous language model that uses transformer encoder architecture, to create contextual embeddings.  
We can access this model using the [Transformers library in HuggingFace]()



In [16]:
# Example sentences
sentence1 = "The supplier agreed to pay the manufacturer"
sentence2 = "The manufacturer agreed to pay the supplier"

In [17]:
# Load BERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_model = AutoModel.from_pretrained('bert-base-uncased')

/Users/amyroberts/opt/anaconda3/lib/python3.8/site-packages/transformers/utils/generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


The first step is to create our tokens 

In [18]:
# Add special tokens and convert to tensor 
input1 = tokenizer(sentence1, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input1['input_ids'][0])

['[CLS]',
 'the',
 'supplier',
 'agreed',
 'to',
 'pay',
 'the',
 'manufacturer',
 '[SEP]']

Notice that the BERT tokenizer added two special tokens `[CLS]` is BERT's way of marking the start of the input document (a sentence in this case). `[SEP]` indicates a separation (such as between sentences) and the end of a document. 

## Try it! 
Create the tokens for sentence 2

In [19]:
# Add special tokens and convert to tensor 
input2 = tokenizer(sentence2, return_tensors="pt", padding=True, truncation=True)

# Take a look at the tokenized sentence
tokenizer.convert_ids_to_tokens(input2['input_ids'][0])

['[CLS]',
 'the',
 'manufacturer',
 'agreed',
 'to',
 'pay',
 'the',
 'supplier',
 '[SEP]']

Let's create a few functions to create the tokens and then use them to create the BERT embeddings at the sentence level and the token level. For now, focus on the outputs. We will dive into how to the details of how code block below works at another time. 

In [20]:
def get_bert_embedding(sentence):
    """
    Get BERT embeddings for a sentence
    """
    # Tokenize and get model outputs
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
    
    # Get model outputs
    outputs = bert_model(**inputs)
    
    # Get embeddings from last hidden state and convert to numpy
    embeddings = outputs.last_hidden_state[0].detach().numpy()
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    return embeddings, tokens



Now we can pass each of our sentences to the function to get the embeddings for each. 

In [21]:
#print each sentence as a reminder of what we are working with
print(sentence1)
print(sentence2)

The supplier agreed to pay the manufacturer
The manufacturer agreed to pay the supplier


In [22]:
# Get embeddings for both sentences
emb1, tokens1 = get_bert_embedding(sentence1)
emb2, tokens2 = get_bert_embedding(sentence2)

In [23]:
def compare_word(word, emb1, tokens1, emb2, tokens2):
    """
    Compare BERT embeddings for a specific word in two embeddings
    """
 
    # Find the word indices
    idx1 = tokens1.index(word)
    idx2 = tokens2.index(word)
    
    # Get the word vectors
    vec1 = emb1[idx1]
    vec2 = emb2[idx2]
    
    # Calculate cosine similarity
    similarity = cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]
    
    print(f"Comparing embeddings for word '{word}':")
    print(f"Similarity score: {similarity:.4f}")
    print(f"Vector in sentence 1 (first 5 values): {vec1[:5]}")
    print(f"Vector in sentence 2 (first 5 values): {vec2[:5]}")
    
    return similarity

In [24]:
supplier_score = compare_word('supplier', emb1, tokens1, emb2, tokens2)


Comparing embeddings for word 'supplier':
Similarity score: 0.7416
Vector in sentence 1 (first 5 values): [ 0.9074207  -0.43041566  0.32430553  0.1308197   0.04717512]
Vector in sentence 2 (first 5 values): [ 0.19103675 -0.46897602  0.00414992  0.13056713 -0.34808046]


In [25]:
manufacturer_score = compare_word('manufacturer', emb1, tokens1, emb2, tokens2)

Comparing embeddings for word 'manufacturer':
Similarity score: 0.8168
Vector in sentence 1 (first 5 values): [ 0.01716887 -0.2678847  -0.25919065  0.195502   -0.4612602 ]
Vector in sentence 2 (first 5 values): [ 0.7152195  -0.4425239   0.26557133  0.18022393 -0.00217662]


Same word but different embeddings! That's because the order in which they were used (sequence) was used in the embedding. 

## Try it! 
Checkout the embeddings for another word of your choice. 

In [26]:
compare_word('pay', emb1, tokens1, emb2, tokens2)

Comparing embeddings for word 'pay':
Similarity score: 0.9909
Vector in sentence 1 (first 5 values): [ 0.17565186 -0.01366385  0.3459662   0.2974596   0.11193749]
Vector in sentence 2 (first 5 values): [ 0.17775811 -0.1218427   0.3169069   0.31182224  0.03333663]


0.9909031

Notice that "pay" has a very high similarity across the two sentences. 

### Summary
In this notebook, we learned how to use: 
1. Word embeddings to represent text data
2. Word2Vec model to generate static word embeddings for a list of reviews
   - Word2Vec is a static embedding that has the same numerical representation for a given word regardless of context 
   - Its performance varies based on how it is tuned and the data used for training 
3. Pre-trained BERT model to generate contextual embeddings for a pair of sentences.
   - BERT is a contextual embedding that creates different embeddings for the same word depending on how it was used in a sentence. 

### Recommended readings 
- [Foundational word2vec paper](https://arxiv.org/abs/1301.3781)  
- [Foundational BERT paper](https://arxiv.org/pdf/1810.04805) 



### 